To align with a company who, build a Fleet Maintenance Management System (FMMS).

A company which focuses on Asset Lifecycle Management. Mostly care about the relationship between parts, labor costs, downtime, and PM (Preventive Maintenance) schedules.

Here is the step-by-step roadmap to build this from scratch, designed to look like a professional Fleet Maintenance Management System (FMMS) industry solution.

**Step 1: The "Physical Logic" Data Generation**

Instead of one flat file, we will create a Relational Structure. Real maintenance data is split across different tables. We will simulate three core tables.

The Plan:

- Assets Table: Static info (Vehicle ID, Year, Make).
- Telemetry Table: Real-time sensor "pings" (Odometer, Fuel Level, Engine Hours).
- Work Orders Table: Historical repairs (Part Replaced, Labor Hours, Cost, Failure Type).



In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

# 1. Assets: 1000 Vehicles
n_vehicles = 1000
df_assets = pd.DataFrame({
    'asset_id': [f'ASSET_{i:03d}' for i in range(n_vehicles)],
    'asset_type': np.random.choice(['Heavy Duty', 'Medium Duty', 'Light Duty'], n_vehicles),
    'purchase_year': np.random.randint(2015, 2023, n_vehicles),
    #'last_service_date': -np.random.randint(0, 100, n_vehicles) # Some were serviced before data starts
})


In [ ]:
print(df_assets.head())
print(df_assets.shape)

**Step 2: Injecting "The Failure Pattern"**

We must hide a failure pattern in the data that isn't obvious.

Logic for your Simulation:

The "Heat" Pattern: If avg_ambient_temp is > 90°F AND engine_load is > 80% for 3 consecutive days, the probability of a "Cooling System" failure in the next 7 days increases by 40%.

The "Vibration" Pattern: Create a vibration_index. If the standard deviation of vibration increases over 5 days, a "Suspension" failure is imminent.

Why this is better:

Consecutive Logic: A single hot day might not break a truck, but a hot day combined with a heavy load is a "Feature Interaction."

Vibration Variance: By making the vibration_index noisier before it fails, will use Rolling Standard Deviation as a feature later.

Multiple Failure Modes: Real fleet systems (like Cetaris) track what failed (Cooling vs. Suspension). This allows eventually do Multi-class classification (predicting the type of failure, not just that it failed).

In [ ]:
import pandas as pd
import numpy as np

# Settings
n_assets = 100
days = 300
data_list = []
work_orders = []


# Set seed for reproducibility 
np.random.seed(42)

for asset_id in range(n_assets):
    a_id = f'ASSET_{asset_id:03d}'
    odometer = np.random.randint(10000, 50000)
    historical_failures = 0
    last_failure_date = None
    last_service_date = -np.random.randint(10, 100) # Initial service history
    total_downtime = 0
    is_down = False
    
    for day in range(days):

        # 1. MAINTENANCE & DOWNTIME CHECK
        # If the asset is "down," it accumulates downtime and doesn't drive miles
        days_since_service = day - last_service_date
        
        if is_down:
            total_downtime += 1
            daily_utilization = 0
            daily_miles = 0
            # Simulating a 3-day repair window
            if last_failure_date is not None and day == (last_failure_date + 3):
                is_down = False
                last_service_date = day # Asset is fresh after repair
        else:
            daily_utilization = np.random.uniform(0.6, 0.9) # 60-90% of the day active
            daily_miles = np.random.randint(100, 400)
        
        odometer += daily_miles

        # 2. BASE TELEMETRY
        #daily_miles = np.random.randint(100, 400)
        #odometer += daily_miles
        ambient_temp = np.random.normal(75, 12) 
        engine_load = np.random.uniform(50, 90)
        vibration_index = np.random.normal(10, 1)
        oil_pressure = np.random.normal(50, 5) # # Normal oil pressure is ~40-60 psi
        coolant_temp = ambient_temp + 120 + np.random.normal(0, 5) # Engine is hotter than air
        #days_since_service = day - last_service_date

                
        # 3. DEGRADATION LOGIC (Symptoms)
        # We check conditions first and "skew" the sensors if the truck is failing
        
        # Heat Degradation: If it's naturally hot/heavy, add extra friction heat
        if ambient_temp > 85 and engine_load > 80:
            ambient_temp += np.random.uniform(5, 15)
            engine_load += np.random.uniform(2, 8)

        # Vibration Degradation: Older assets start failing at the end of the simulation
        if day > 200:
            # Gradually increase vibration as time goes on to simulate wear
            vibration_index += (day - 200) * 0.05 + np.random.normal(0, 2)
 
        # Degradation for Oil: If vibration is high, oil pressure might fluctuate or drop
        if vibration_index > 15:
            oil_pressure -= np.random.uniform(5, 15)
            
        # 4. FAILURE TRIGGERS (Work Orders)
        # Now we check the "Symptomatic" values to see if a Work Order is generated
        # Only check for new failures if the asset is currently running
        if not is_down and day > 10:
        
            # Pattern A: Cooling System Failure
            if ambient_temp > 95 and engine_load > 85:
                if np.random.rand() < 0.03: # Low probability to keep target around 15%
                    is_down = True
                    last_failure_date = day
                    historical_failures += 1
                    work_orders.append({
                        'asset_id': a_id, 
                        'date': day,  #+ 2, # Mechanic arrives 2 days later
                        'error_type': 'Cooling System'
                        #'repair_cost': np.random.uniform(2000, 5000) # keeping onside for now
                    })
            
            # Pattern B: Suspension Failure
            if vibration_index > 18:
                if np.random.rand() < 0.05:
                    is_down = True
                    last_failure_date = day
                    historical_failures += 1
                    work_orders.append({
                        'asset_id': a_id, 
                        'date': day + 1, 
                        'error_type': 'Suspension'
                        #'repair_cost': np.random.uniform(2000, 5000) # keeping onside for now
                    })

        # 5. SAVE TO MASTER LIST
        # This saves the data *after* the degradation was applied
        data_list.append({
            'date': day,
            'asset_id': a_id,
            'odometer': odometer,
            'ambient_temp': ambient_temp,
            'coolant_temp': coolant_temp,
            'oil_pressure': oil_pressure,
            'engine_load': engine_load,
            'vibration_index': vibration_index,
            'daily_utilization': daily_utilization,
            'days_since_service': days_since_service, # Added per objective
            'total_downtime_history': total_downtime,
            'historical_failure_count': historical_failures # Added per objective
        })

# Create DataFrames
df_telemetry = pd.DataFrame(data_list)
df_work_orders = pd.DataFrame(work_orders).drop_duplicates(subset=['asset_id', 'date'])

print(f"✅ Success!")
print(f"Telemetry Rows: {len(df_telemetry)}")
print(f"Work Orders Generated: {len(df_work_orders)}")
print(f"Average Work Orders per Asset: {len(df_work_orders)/n_assets:.2f}")

In [ ]:
# Save generated data to the data/raw folder
import os 

os.makedirs('../data/raw', exist_ok=True)
df_assets.to_csv('../data/raw/assets.csv', index=False)
df_telemetry.to_csv('../data/raw/telemetry.csv', index=False)
df_work_orders.to_csv('../data/raw/work_orders.csv', index=False)


The "Merging" Phase

Now that we have the data, next notebook should be about merging and EDA.

The Goal: need to map the work_orders back to the telemetry so every row of telemetry knows: "Am I within 30 days of a failure?"

Action Item:

- Merge df_telemetry and df_work_orders on asset_id.
- Create a column days_until_failure = failure_date - current_date.
- Create the Target: target = 1 if days_until_failure <= 30 else 0.